This will currently import and download all documents from General Court 194 and will have a commented version that will import and download prior general court documents.

In [45]:
import requests
import pandas as pd
import openpyxl
import time

In [27]:

def documentImport(general_court):
    try:
        response = requests.get("https://malegislature.gov/api/GeneralCourts/"+ str(general_court) +"/Documents/")
        response.raise_for_status()  # Raises an HTTPError for 4xx/5xx status codes e.g. 404 file not found
        
        #Turn into a dataframe
        df = pd.json_normalize(response.json())
        
        #There are a few blank columns every time so these will be removed
        try:
            df.drop('JointSponsor', axis=1,inplace=True) 
            df.drop('JointSponsor.Id', axis=1,inplace=True)
            df.drop('JointSponsor.Name', axis=1,inplace=True)
            df.drop('JointSponsor.Type', axis=1,inplace=True)
            df.drop('JointSponsor.Details', axis=1,inplace=True)
            df.drop('JointSponsor.ResponseDate', axis=1,inplace=True)
        finally:
            try:
                df.drop('PrimarySponsor', axis=1,inplace=True) #This one is just an artifact based on how it reads the json file.
            finally:
                return df
    except requests.exceptions.HTTPError as err:
        print(f"HTTP Error: {err}")
    except requests.exceptions.ConnectionError as err:
        print(f"Connection Error: {err}")
    except requests.exceptions.Timeout as err:
        print(f"Timeout Error: {err}")
    except requests.exceptions.RequestException as err:
        print(f"An unexpected error occurred: {err}")
    

In [77]:
def getExtraDocumentInfo(url, general_court): 
    try:
        resp = requests.get(url, timeout=8)
        resp.raise_for_status()
        data = resp.json()
        description = data["DocumentText"]
        rollCalls = data["RollCalls"]
        committee = data["CommitteeRecommendations"]
        amendments = data["Amendments"]
        return(description, rollCalls, committee,amendments)
    
    except Exception as e:
        
        print(f"⚠ Error fetching: {e}")
        return(e, e, e,e)

In [78]:
def addExtraInfo(df, general_court):
    df2 = pd.DataFrame(columns=["Description", "RollCalls", 'Committee',"Amendments"])
    for i in range(len(df)):
        if (df["Details"][i]) is None:
            desc, rollcall, committee, amendments = "NaN","NaN","NaN","NaN"
        else:
            desc, rollcall, committee, amendments = getExtraDocumentInfo(df["Details"][i], general_court)
        df2.loc[len(df2)] = {"Description":desc , "RollCalls":rollcall, 'Committee':committee,"Amendments":amendments}
    merged_df = df.join(df2)
    return merged_df

In [79]:

def fullDataImport():
    df = pd.DataFrame()
    cont = True
    generalCourt = 183 #The first general court recorded on the API from testing
    start_time = time.time()
    while True:
        try:
            imported_df = documentImport(generalCourt)
            merged_df = addExtraInfo(imported_df, generalCourt)
            df = pd.concat([df, merged_df], ignore_index=True)
            current_time = time.time()
            elapsed_time = current_time - start_time
            print("Finished General Court: " +str(generalCourt)+" in " +str(elapsed_time) +" Seconds")
            generalCourt += 1 
        except Exception:
            break
    return (df)
        

In [80]:
#Haven't Yet managed to make it do all documents, Note General Court 183-185 have very few documents so they are faster.


#df = fullDataImport() 


#Just gets a specific generalCourt instead
df1 = documentImport(194)
df = addExtraInfo(df1, 194)

⚠ Error fetching: 400 Client Error: The requested Document could not be found in General Court "194". for url: https://malegislature.gov/api/GeneralCourts/194/Documents/SD435
⚠ Error fetching: 400 Client Error: The requested Document could not be found in General Court "194". for url: https://malegislature.gov/api/GeneralCourts/194/Documents/SD436
⚠ Error fetching: 400 Client Error: The requested Document could not be found in General Court "194". for url: https://malegislature.gov/api/GeneralCourts/194/Documents/SD239
⚠ Error fetching: 400 Client Error: The requested Document could not be found in General Court "194". for url: https://malegislature.gov/api/GeneralCourts/194/Documents/SD98
⚠ Error fetching: 400 Client Error: The requested Document could not be found in General Court "194". for url: https://malegislature.gov/api/GeneralCourts/194/Documents/SD1422
⚠ Error fetching: 400 Client Error: The requested Document could not be found in General Court "194". for url: https://malegi

In [84]:

print(df.head())

  BillNumber DocketNumber                                              Title  \
0      H3444       HD2050            An Act relative to s-license compliance   
1      H1283       HD3832    An Act relative to real estate disclosure forms   
2      H1399       HD3446  An Act relative to an individual Medicare mark...   
3      H1282       HD3438  An Act relative to the Massachusetts Uniform C...   
4      H1932       HD3437  An Act relative to the uniform voidable transa...   

                                          Cosponsors  GeneralCourtNumber  \
0  [{'Id': 'JCD1', 'Name': 'James C. Arena-DeRosa...                 194   
1  [{'Id': 'AHP1', 'Name': 'Alice Hanlon Peisch',...                 194   
2  [{'Id': 'AHP1', 'Name': 'Alice Hanlon Peisch',...                 194   
3  [{'Id': 'AHP1', 'Name': 'Alice Hanlon Peisch',...                 194   
4  [{'Id': 'AHP1', 'Name': 'Alice Hanlon Peisch',...                 194   

                                             Details  IsDocket

In [ ]:
#df3 = addExtraInfo(df[:10].copy(), 183)

In [ ]:
#This could be uploaded to SQL to get it to the looker studio system if needed instead of downloading as xlsx but might not be formatted correctly
#df3.to_sql("output_dataframe.xlsx", con = engine ,index=False) 


In [ ]:
df.to_csv("data/data_Bills.csv", index = False)

# It bugged out on excel and might bug out on other ones too.

In [ ]:
#Example of reaing the excel file.
#df4 = pd.read_excel("data/data_Bills.xlsx")
#print(df4.head())

In [ ]:
#df5 = pd.read_csv("data/data_Bills.csv")